# MCA-YOLO-A Exp2 云端训练

训练 MobileNetV3 + Coordinate Attention，不加入Alpha-IoU或P2。

In [ ]:
# 1. 检查GPU并安装依赖
import torch
print(torch.__version__, torch.cuda.is_available())
if not torch.cuda.is_available(): raise RuntimeError('请在Colab中选择T4 GPU')
!pip install ultralytics==8.4.106 -q


In [ ]:
# 2. 拉取项目并配置云端数据路径
import os
import sys
REPO_PATH = '/content/mca-yolo-reproduce'
if not os.path.exists(REPO_PATH):
    !git clone https://github.com/Cranzz/mca-yolo-reproduce.git {REPO_PATH}
else:
    !cd {REPO_PATH} && git pull
%cd {REPO_PATH}
yaml_path = f'{REPO_PATH}/data/rdd2022.yaml'
with open(yaml_path, 'w') as f:
    f.write(f'''path: {REPO_PATH}/data/yolo_format\ntrain: train/images\nval: val/images\ntest: test/images\nnc: 4\nnames: ['D00', 'D10', 'D20', 'D40']''')


In [ ]:
# 3. 注册MobileNetV3CA并构建模型
sys.path.insert(0, REPO_PATH)
from models.modules import CoordinateAttention, MobileNetV3CA
from ultralytics import YOLO
from ultralytics.nn import tasks
tasks.MobileNetV3CA = MobileNetV3CA
tasks.TorchVision = MobileNetV3CA
model = YOLO(f'{REPO_PATH}/models/yolov8n_mobilenetv3_ca.yaml')
ca_count = sum(isinstance(m, CoordinateAttention) for m in model.model.modules())
print(f'CA模块数量: {ca_count}')
if ca_count == 0: raise RuntimeError('未注册CA模块')


In [ ]:
# 4. 训练Exp2
results = model.train(
    data=yaml_path, epochs=100, imgsz=640, batch=16,
    name='yolov8n_rdd2022_exp2_mobilenetv3_ca_colab', device=0,
    optimizer='SGD', lr0=0.01, momentum=0.937,
    plots=True,
)
print(f'最佳模型: {results.save_dir}')


In [ ]:
# 5. 测试集评估
metrics = model.val(data=yaml_path, split='test', imgsz=640, batch=16, device=0)
print(f'mAP50: {metrics.box.map50:.4f}')
print(f'mAP50-95: {metrics.box.map:.4f}')
